# VibeVoice-Realtime — Local Windows Demo (V01)

Adapted from [microsoft/VibeVoice](https://github.com/microsoft/VibeVoice/blob/main/demo/vibevoice_realtime_colab.ipynb) for local execution on Windows.

**Requirements**
- CUDA-capable GPU (VRAM ≥ 8 GB recommended)
- Python 3.10 environment with `pip`
- `git` available on PATH
- Internet access for model download from Hugging Face

**Steps**
1. Setup environment (clone repo, install deps, download model)
2. (Optional) HuggingFace login if download stalls
3. (Optional) Download experimental voices
4. Launch the VibeVoice-Realtime demo server and open the local URL

## Step 1: Setup Environment

In [1]:
# --- GPU check ---
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU detected: {gpu_name}")
    if 'T4' not in gpu_name:
        print("   (Not a T4 — the demo was written for T4 but any CUDA GPU should work)")
else:
    print("""
⚠️  WARNING: No CUDA GPU detected.
    VibeVoice-Realtime requires a CUDA GPU for real-time performance.
    Make sure:
      • CUDA drivers are installed
      • This kernel uses a CUDA-enabled PyTorch build
      • The correct conda / venv environment is selected as the kernel
    """)

✅ GPU detected: NVIDIA GeForce RTX 3090
   (Not a T4 — the demo was written for T4 but any CUDA GPU should work)


In [2]:
import os
from pathlib import Path

# ── Configuration ────────────────────────────────────────────────────────────
# Root folder where the repo is cloned and the model is stored.
# Change this if you prefer a different location.
BASE_DIR   = Path(r"D:\VibeVoice")
REPO_DIR   = BASE_DIR / "VibeVoice"
MODEL_DIR  = BASE_DIR / "models" / "VibeVoice-Realtime-0.5B"
DEMO_PORT  = 8000
# ─────────────────────────────────────────────────────────────────────────────

BASE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Clone repository (skip if already present)
if not (REPO_DIR / ".git").exists():
    ret = os.system(
        f'git clone --quiet --branch main --depth 1 '
        f'https://github.com/microsoft/VibeVoice.git "{REPO_DIR}"'
    )
    print("✅ Cloned VibeVoice repository" if ret == 0 else "❌ git clone failed")
else:
    print(f"✅ Repository already present at {REPO_DIR}")

# Install project dependencies (streamingtts extra)
ret = os.system(f'pip install --quiet -e "{REPO_DIR}[streamingtts]"')
print("✅ Installed dependencies" if ret == 0 else "❌ pip install failed")

✅ Cloned VibeVoice repository
✅ Installed dependencies


In [3]:
from pathlib import Path
from huggingface_hub import snapshot_download

# BASE_DIR / MODEL_DIR defined in the cell above.
# Re-define here in case cells are run out of order.
BASE_DIR  = Path(r"D:\VibeVoice")
MODEL_DIR = BASE_DIR / "models" / "VibeVoice-Realtime-0.5B"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

snapshot_download(
    "microsoft/VibeVoice-Realtime-0.5B",
    local_dir=str(MODEL_DIR),
)
print("✅ Downloaded model: microsoft/VibeVoice-Realtime-0.5B")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/360 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

figures/Fig1.png:   0%|          | 0.00/124k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.04G [00:00<?, ?B/s]

✅ Downloaded model: microsoft/VibeVoice-Realtime-0.5B


### [Optional] HuggingFace Login

If the download above stalls (> 1 minute), interrupt it, run the cell below to log in, then re-run the download cell.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Retry download after login
from pathlib import Path
from huggingface_hub import snapshot_download

BASE_DIR  = Path(r"D:\VibeVoice")
MODEL_DIR = BASE_DIR / "models" / "VibeVoice-Realtime-0.5B"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

snapshot_download(
    "microsoft/VibeVoice-Realtime-0.5B",
    local_dir=str(MODEL_DIR),
)
print("✅ Downloaded model: microsoft/VibeVoice-Realtime-0.5B")

### [Optional] Download Experimental Voices

In [ ]:
import os
from pathlib import Path

BASE_DIR = Path(r"D:\VibeVoice")
REPO_DIR = BASE_DIR / "VibeVoice"
script   = REPO_DIR / "demo" / "download_experimental_voices.sh"

if script.exists():
    # Requires Git Bash or WSL
    ret = os.system(f'bash "{script}"')
    print("✅ Experimental voices downloaded" if ret == 0 else "❌ Script failed — make sure bash (Git Bash / WSL) is on PATH")
else:
    print(f"❌ Script not found: {script}")

## Step 2: Launch VibeVoice-Realtime Demo

This starts the demo server on `http://localhost:<PORT>`.  
Open that URL in your browser after the **"Uvicorn running on"** message appears.

> ℹ️ Unlike the Colab version, no `cloudflared` tunnel is needed — access is direct via `localhost`.

In [11]:
import subprocess
import threading
import time
from pathlib import Path

BASE_DIR  = Path(r"D:\VibeVoice")
REPO_DIR  = BASE_DIR / "VibeVoice"
MODEL_DIR = BASE_DIR / "models" / "VibeVoice-Realtime-0.5B"
DEMO_PORT = 8000

demo_script = REPO_DIR / "demo" / "vibevoice_realtime_demo.py"

cmd = (
    f'python "{demo_script}" '
    f'--model_path "{MODEL_DIR}" '
    f'--port {DEMO_PORT}'
)
print(f"Starting: {cmd}\n")

srv = subprocess.Popen(
    cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True,
)

server_ready = False

def read_srv():
    global server_ready
    for ln in srv.stdout:
        print(ln.strip())
        if "Uvicorn running on" in ln or "Application startup complete" in ln:
            server_ready = True

threading.Thread(target=read_srv, daemon=True).start()

# Wait for server to be ready (up to 60 s)
for _ in range(240):
    if server_ready:
        break
    time.sleep(0.25)

if server_ready:
    print(f"\n✅ Demo server ready — open http://localhost:{DEMO_PORT} in your browser")
else:
    print("\n⚠️  Server did not report ready within 60 s — check the output above for errors.")

Starting: python "D:\VibeVoice\VibeVoice\demo\vibevoice_realtime_demo.py" --model_path "D:\VibeVoice\models\VibeVoice-Realtime-0.5B" --port 8000

INFO:     Started server process [72132]
INFO:     Waiting for application startup.
[startup] Loading processor from D:\VibeVoice\models\VibeVoice-Realtime-0.5B
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization.
The tokenizer class you load from this checkpoint is 'Qwen2Tokenizer'.
The class this function is called from is 'VibeVoiceTextTokenizerFast'.
Using device: cuda, torch_dtype: torch.bfloat16, attn_implementation: flash_attention_2
Error loading the model. Trying to use SDPA. However, note that only flash_attention_2 has been fully tested, and using SDPA may result in lower audio quality.
Some weights of VibeVoiceStreamingForConditionalGenerationInference were not initialized from the model checkpoint at D:\VibeVoice\models\VibeVoic

### Stop the server

Run the cell below to terminate the demo server process when you are done.

In [5]:
try:
    srv.terminate()
    srv.wait(timeout=5)
    print("✅ Server stopped.")
except Exception as e:
    print(f"Could not stop server: {e}")

✅ Server stopped.


### Transcribe a local WAV file

Run the next cell to transcribe `d:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\2165.wav` using the local VibeVoice-Realtime setup.

In [16]:
from pathlib import Path

AUDIO_FILE = Path(r"d:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\2165.wav")
BASE_URL = "http://127.0.0.1:8000"

if not AUDIO_FILE.exists():
    print(f"❌ Audio file not found: {AUDIO_FILE}")
else:
    print("This page is a TTS demo, not a speech-to-text upload page.")
    print("It accepts text input and generates speech.")
    print("\nSo this demo cannot transcribe your WAV file from the browser UI.")
    print(f"Target file:\n{AUDIO_FILE}")
    print("\nIf you want, I can add a new diagnostic/code cell that inspects the local VibeVoice source and tries direct Python transcription instead.")

This page is a TTS demo, not a speech-to-text upload page.
It accepts text input and generates speech.

So this demo cannot transcribe your WAV file from the browser UI.
Target file:
d:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\2165.wav

If you want, I can add a new diagnostic/code cell that inspects the local VibeVoice source and tries direct Python transcription instead.


### Transcribe the WAV file with `faster-whisper`

Run the next cell to install `faster-whisper` if needed and transcribe `d:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\2165.wav`.

In [17]:
import importlib
import subprocess
import sys
from pathlib import Path

AUDIO_FILE = Path(r"d:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\2165.wav")
MODEL_SIZE = "small"

if not AUDIO_FILE.exists():
    print(f"❌ Audio file not found: {AUDIO_FILE}")
else:
    if importlib.util.find_spec("faster_whisper") is None:
        print("Installing faster-whisper...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "faster-whisper"])

    import torch
    from faster_whisper import WhisperModel

    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"

    print(f"Loading model '{MODEL_SIZE}' on {device} ({compute_type})...")
    model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute_type)

    segments, info = model.transcribe(str(AUDIO_FILE), beam_size=5)
    text = " ".join(segment.text.strip() for segment in segments).strip()

    print("✅ Transcription completed")
    print(f"Detected language: {info.language} (probability={info.language_probability:.3f})")
    print("\nTranscript:\n")
    print(text if text else "[No transcript returned]")

Loading model 'small' on cuda (float16)...


vocabulary.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/484M [00:00<?, ?B/s]

✅ Transcription completed
Detected language: nl (probability=0.989)

Transcript:

Het is de eerste keer dat u hier bent of bent u er vaker geweest? Nou, ik heb niet vaker geweest. 30 jaar, dus... Oké, maar nog maar eens voor deze klachten, denk ik. Nee, in deze klachten heb ik al 10 jaar, denk ik. Oeh, dat is lang. Ja, en ik heb een nieuwe huppen gezet en dan dachten ze dat het weggaan is met huppen, maar het is niet weggegaan. Ja, met wat? En het is moeilijk in de auto te stappen, het is moeilijk om op te tillen benen. En als ik ergens kom, als dat niet gelijk is, dan stoot ik het weer op. Nou, het is zo pijn. Ja, wat was het? Ik denk dat ik iets moet gaan doen. Ja, snap ik. Nou, hartstikke goed. In ieder geval. Na 10 jaar, ja. Want even kijken, want u heeft last van de lies, of alle beide kant op de ene kant? Nee, een beetje links, ja. Linker lies heeft u last van. En daar heeft u nu 10 jaar al last van. En hoe lang heeft u nu al de nieuwe huppen? Ik heb die linker, heb ik al 6 jaar,

### Sentence timestamps

Run the next cell to transcribe the WAV file and print sentence-level timestamps in the output.

In [18]:
import re
from pathlib import Path

AUDIO_FILE = Path(r"d:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\2165.wav")
MODEL_SIZE = "small"

if not AUDIO_FILE.exists():
    print(f"❌ Audio file not found: {AUDIO_FILE}")
else:
    import torch
    from faster_whisper import WhisperModel

    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"

    def format_ts(seconds: float) -> str:
        total_ms = int(round(seconds * 1000))
        hours, rem = divmod(total_ms, 3600_000)
        minutes, rem = divmod(rem, 60_000)
        secs, ms = divmod(rem, 1000)
        if hours:
            return f"{hours:02d}:{minutes:02d}:{secs:02d}.{ms:03d}"
        return f"{minutes:02d}:{secs:02d}.{ms:03d}"

    def split_sentences_with_timestamps(segment_list):
        sentence_end = re.compile(r"(.+?[.!?]+(?:[\"'”’\)\]]+)?)($|\s+)", re.DOTALL)
        rows = []

        for seg in segment_list:
            seg_text = seg.text.strip()
            if not seg_text:
                continue

            matches = list(sentence_end.finditer(seg_text))
            if not matches:
                rows.append((seg.start, seg.end, seg_text))
                continue

            consumed = 0
            parts = []
            for m in matches:
                sent = m.group(1).strip()
                if sent:
                    parts.append(sent)
                    consumed = m.end()

            tail = seg_text[consumed:].strip()
            if tail:
                parts.append(tail)

            total_chars = sum(len(p) for p in parts) or 1
            current_start = seg.start
            duration = max(seg.end - seg.start, 0)

            for i, sent in enumerate(parts):
                portion = len(sent) / total_chars
                if i == len(parts) - 1:
                    current_end = seg.end
                else:
                    current_end = min(seg.end, current_start + duration * portion)
                rows.append((current_start, current_end, sent))
                current_start = current_end

        return rows

    print(f"Loading model '{MODEL_SIZE}' on {device} ({compute_type})...")
    model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute_type)

    segments, info = model.transcribe(str(AUDIO_FILE), beam_size=5)
    segment_list = list(segments)
    sentence_rows = split_sentences_with_timestamps(segment_list)

    print("✅ Transcription with timestamps completed")
    print(f"Detected language: {info.language} (probability={info.language_probability:.3f})")
    print("\nSentence-level transcript:\n")

    for start, end, sent in sentence_rows:
        print(f"[{format_ts(start)} --> {format_ts(end)}] {sent}")

Loading model 'small' on cuda (float16)...
✅ Transcription with timestamps completed
Detected language: nl (probability=0.989)

Sentence-level transcript:

[00:00.000 --> 00:03.680] Het is de eerste keer dat u hier bent of bent u er vaker geweest?
[00:03.680 --> 00:06.000] Nou, ik heb niet vaker geweest.
[00:06.880 --> 00:08.760] 30 jaar, dus...
[00:08.760 --> 00:11.800] Oké, maar nog maar eens voor deze klachten, denk ik.
[00:11.800 --> 00:15.640] Nee, in deze klachten heb ik al 10 jaar, denk ik.
[00:15.640 --> 00:16.720] Oeh, dat is lang.
[00:16.720 --> 00:23.360] Ja, en ik heb een nieuwe huppen gezet en dan dachten ze dat het weggaan is met huppen, maar het is niet weggegaan.
[00:23.360 --> 00:24.480] Ja, met wat?
[00:24.480 --> 00:31.600] En het is moeilijk in de auto te stappen, het is moeilijk om op te tillen benen.
[00:31.600 --> 00:36.040] En als ik ergens kom, als dat niet gelijk is, dan stoot ik het weer op.
[00:36.040 --> 00:38.240] Nou, het is zo pijn.
[00:38.240 --> 00:38.

### Two-speaker diarization

Run the next cell to produce a simple two-speaker dialogue view with timestamps. This uses an alternating-speaker heuristic over timestamped sentence chunks, not a true speaker-embedding diarization model.

In [ ]:
import re
from pathlib import Path

AUDIO_FILE = Path(r"d:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\2165.wav")
MODEL_SIZE = "small"

if not AUDIO_FILE.exists():
    print(f"❌ Audio file not found: {AUDIO_FILE}")
else:
    import torch
    from faster_whisper import WhisperModel

    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"

    def format_ts(seconds: float) -> str:
        total_ms = int(round(seconds * 1000))
        hours, rem = divmod(total_ms, 3600_000)
        minutes, rem = divmod(rem, 60_000)
        secs, ms = divmod(rem, 1000)
        if hours:
            return f"{hours:02d}:{minutes:02d}:{secs:02d}.{ms:03d}"
        return f"{minutes:02d}:{secs:02d}.{ms:03d}"

    def split_sentences_with_timestamps(segment_list):
        sentence_end = re.compile(r"(.+?[.!?]+(?:[\"'”’\)\]]+)?)($|\s+)", re.DOTALL)
        rows = []

        for seg in segment_list:
            seg_text = seg.text.strip()
            if not seg_text:
                continue

            matches = list(sentence_end.finditer(seg_text))
            if not matches:
                rows.append((seg.start, seg.end, seg_text))
                continue

            consumed = 0
            parts = []
            for m in matches:
                sent = m.group(1).strip()
                if sent:
                    parts.append(sent)
                    consumed = m.end()

            tail = seg_text[consumed:].strip()
            if tail:
                parts.append(tail)

            total_chars = sum(len(p) for p in parts) or 1
            current_start = seg.start
            duration = max(seg.end - seg.start, 0)

            for i, sent in enumerate(parts):
                portion = len(sent) / total_chars
                if i == len(parts) - 1:
                    current_end = seg.end
                else:
                    current_end = min(seg.end, current_start + duration * portion)
                rows.append((current_start, current_end, sent))
                current_start = current_end

        return rows

    def assign_two_speakers(sentence_rows):
        labeled = []
        current_speaker = 1
        previous_end = None

        for start, end, sent in sentence_rows:
            if previous_end is not None:
                pause = start - previous_end
                if pause > 0.8:
                    current_speaker = 2 if current_speaker == 1 else 1
            labeled.append((f"Speaker {current_speaker}", start, end, sent))
            previous_end = end

        return labeled

    print(f"Loading model '{MODEL_SIZE}' on {device} ({compute_type})...")
    model = WhisperModel(MODEL_SIZE, device=device, compute_type=compute_type)

    segments, info = model.transcribe(str(AUDIO_FILE), beam_size=5)
    segment_list = list(segments)
    sentence_rows = split_sentences_with_timestamps(segment_list)
    speaker_rows = assign_two_speakers(sentence_rows)

    print("✅ Two-speaker dialogue view completed")
    print(f"Detected language: {info.language} (probability={info.language_probability:.3f})")
    print("\nDialogue transcript:\n")

    for speaker, start, end, sent in speaker_rows:
        print(f"[{format_ts(start)} --> {format_ts(end)}] {speaker}: {sent}")